# 🫀 Fitbit Charge 5 ECG → Arrhythmia Classifier

This notebook demonstrates the full pipeline:
1. **Fetch** real ECG waveform data from Fitbit Web API (Charge 5)
2. **Parse** the raw `waveformSamples` into a numpy signal
3. **Visualise** the waveform
4. **Classify** using our trained Random Forest model
5. **Interpret** the risk output

> ⚙️ **Setup required:** Add your Fitbit OAuth2 access token in the cell below.  
> Get one at https://dev.fitbit.com/build/reference/web-api/developer-guide/getting-started/

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import json
from datetime import datetime, timedelta
from scipy.signal import butter, filtfilt

from src.features.extractor import hand_crafted
from src.models.classifier import ArrhythmiaClassifier

print('✅ Imports OK')

## 1. Fitbit API 연결 설정

**액세스 토큰 발급 방법 (5분)**
1. https://dev.fitbit.com 에서 무료 앱 등록
2. OAuth 2.0 Tutorial 페이지에서 토큰 발급
3. 아래 `ACCESS_TOKEN`에 붙여넣기

In [ ]:
# ── 여기에 본인 토큰 입력 ──────────────────────────────
ACCESS_TOKEN = "YOUR_FITBIT_ACCESS_TOKEN_HERE"
# ──────────────────────────────────────────────────────

HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
ECG_URL = "https://api.fitbit.com/1/user/-/ecg/list.json"

# Fitbit ECG API 파라미터
after_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S")
params = {
    "afterDate": after_date,
    "sort": "desc",
    "limit": 10,
    "offset": 0,
}

print(f"Querying ECG records after: {after_date}")

## 2. ECG 데이터 가져오기

> 토큰이 없으면 아래 **"Demo Mode"** 셀을 대신 실행하세요.

In [ ]:
# ── 실제 Fitbit API 호출 ───────────────────────────────
# (토큰이 없으면 다음 셀의 Demo Mode를 사용하세요)

def fetch_ecg_readings(headers, params):
    resp = requests.get(ECG_URL, headers=headers, params=params)
    
    if resp.status_code == 401:
        print("❌ 인증 오류 — 토큰을 확인하거나 Demo Mode를 사용하세요")
        return None
    if resp.status_code != 200:
        print(f"❌ API 오류: {resp.status_code} — {resp.text}")
        return None
    
    data = resp.json()
    readings = data.get("ecgReadings", [])
    print(f"✅ {len(readings)}개 ECG 기록 발견")
    for i, r in enumerate(readings):
        print(f"  [{i}] {r['startTime']} | {r['resultClassification']} "
              f"| HR: {r['averageHeartRate']} bpm "
              f"| {r['numberOfWaveformSamples']} samples")
    return readings

readings = fetch_ecg_readings(HEADERS, params)

In [ ]:
# ── Demo Mode: 토큰 없이 실행 ─────────────────────────
# 실제 Fitbit API 응답 구조와 동일한 형식의 합성 데이터

def make_demo_readings():
    from src.data.generator import _generate_normal_beat, _generate_veb_beat, BEAT_LEN
    
    def beat_to_samples(beat_fn, n_beats=10):
        """여러 박자를 이어붙여 30초 분량(7500 samples @ 250Hz) 생성"""
        # Fitbit은 250Hz, 30초 = 7500 samples
        samples = []
        for _ in range(n_beats):
            beat = beat_fn(noise_std=0.02)
            # 360→250Hz 리샘플 (단순 보간)
            x_old = np.linspace(0, 1, len(beat))
            x_new = np.linspace(0, 1, 250)
            resampled = np.interp(x_new, x_old, beat)
            samples.extend((resampled * 10922).astype(int).tolist())
        # 7500개로 맞추기
        samples = samples[:7500]
        samples += [0] * (7500 - len(samples))
        return samples

    return [
        {
            "startTime": "2026-04-23T09:00:00.000",
            "averageHeartRate": 68,
            "resultClassification": "Normal Sinus Rhythm",
            "waveformSamples": beat_to_samples(_generate_normal_beat),
            "samplingFrequencyHz": "250",
            "scalingFactor": 10922,
            "numberOfWaveformSamples": 7500,
        },
        {
            "startTime": "2026-04-23T10:30:00.000",
            "averageHeartRate": 72,
            "resultClassification": "Inconclusive",
            "waveformSamples": beat_to_samples(_generate_veb_beat),
            "samplingFrequencyHz": "250",
            "scalingFactor": 10922,
            "numberOfWaveformSamples": 7500,
        },
    ]

# 토큰이 없거나 readings가 None이면 demo 사용
if not readings:
    readings = make_demo_readings()
    print(f"🔬 Demo Mode: {len(readings)}개 합성 ECG 기록 사용")
    for i, r in enumerate(readings):
        print(f"  [{i}] {r['startTime']} | {r['resultClassification']} "
              f"| HR: {r['averageHeartRate']} bpm")

## 3. 파형 파싱 및 전처리

In [ ]:
def parse_waveform(reading):
    """
    Fitbit API 응답에서 mV 단위 신호로 변환.
    
    Fitbit은 정수로 인코딩된 값을 반환.
    실제 전압 = sample / scalingFactor (mV 근사)
    """
    raw     = np.array(reading["waveformSamples"], dtype=np.float32)
    scale   = float(reading["scalingFactor"])
    fs      = float(reading["samplingFrequencyHz"])
    signal  = raw / scale
    return signal, fs


def bandpass_filter(signal, fs, low=0.5, high=40.0):
    """
    0.5–40 Hz 대역통과 필터 — 기저선 드리프트와 고주파 노이즈 제거.
    임상 ECG 표준 필터 대역폭.
    """
    nyq = fs / 2
    b, a = butter(2, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal)


def segment_beats(signal, fs, min_rr_ms=300):
    """
    R-peak 검출 후 1초 창으로 개별 박자 분리.
    """
    from scipy.signal import find_peaks
    min_dist = int(fs * min_rr_ms / 1000)
    peaks, _ = find_peaks(signal, distance=min_dist, height=signal.max() * 0.4)
    
    half = int(fs * 0.5)  # R-peak 전후 0.5초
    beat_len = int(fs)    # 총 1초
    beats = []
    for p in peaks:
        start, end = p - half, p + half
        if start >= 0 and end <= len(signal):
            beat = signal[start:end]
            # 360샘플로 리샘플 (모델 입력 형식)
            x_old = np.linspace(0, 1, len(beat))
            x_new = np.linspace(0, 1, 360)
            beat_resampled = np.interp(x_new, x_old, beat)
            beats.append(beat_resampled)
    return np.array(beats, dtype=np.float32), peaks

print("✅ 전처리 함수 정의 완료")

## 4. 전체 파형 시각화

In [ ]:
fig, axes = plt.subplots(len(readings), 1, figsize=(14, 3.5 * len(readings)))
if len(readings) == 1:
    axes = [axes]

processed = []  # (signal, fs, beats, peaks, reading) 저장

for ax, reading in zip(axes, readings):
    signal, fs = parse_waveform(reading)
    signal_f   = bandpass_filter(signal, fs)
    beats, peaks = segment_beats(signal_f, fs)
    
    t = np.arange(len(signal_f)) / fs
    ax.plot(t, signal_f, color="#2c3e50", lw=0.8, label="ECG (filtered)")
    ax.scatter(peaks / fs, signal_f[peaks],
               color="#e74c3c", s=40, zorder=5, label="R-peaks")
    
    fitbit_label = reading["resultClassification"]
    color = "#27ae60" if "Normal" in fitbit_label else "#e67e22"
    ax.set_title(
        f"{reading['startTime']}  |  Fitbit: {fitbit_label}  "
        f"|  HR: {reading['averageHeartRate']} bpm  "
        f"|  {len(beats)} beats detected",
        fontsize=10, color=color
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (mV)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    processed.append((signal_f, fs, beats, peaks, reading))

plt.suptitle("Fitbit Charge 5 — ECG Recordings", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fitbit_ecg_waveforms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → fitbit_ecg_waveforms.png")

## 5. 분류 모델로 박자별 위험도 분석

In [ ]:
# 모델 로드
clf = ArrhythmiaClassifier.load("output/model.pkl")

LABEL_NAMES = {0: "N — Normal", 1: "S — SVEB",   2: "V — VEB"}
RISK_EMOJI  = {0: "✅",          1: "⚠️ ",          2: "🚨"}
RISK_COLOR  = {0: "#27ae60",    1: "#e67e22",     2: "#e74c3c"}

all_results = []

for signal_f, fs, beats, peaks, reading in processed:
    if len(beats) == 0:
        print(f"⚠️  {reading['startTime']}: 박자 감지 실패")
        continue
    
    feats = hand_crafted(beats)
    preds = clf.predict(feats)
    probas = clf.predict_proba(feats)
    
    # 세션 요약
    unique, counts = np.unique(preds, return_counts=True)
    summary = {int(u): int(c) for u, c in zip(unique, counts)}
    veb_ratio = summary.get(2, 0) / len(preds) * 100
    
    session_risk = "🚨 HIGH" if veb_ratio > 10 else ("⚠️  MODERATE" if summary.get(1, 0) > 0 else "✅ LOW")
    
    print(f"\n{'═'*60}")
    print(f"📅 {reading['startTime']}")
    print(f"   Fitbit 판정: {reading['resultClassification']}")
    print(f"   총 박자 수:  {len(preds)}개")
    print(f"   분류 결과:   N={summary.get(0,0)}  S={summary.get(1,0)}  V={summary.get(2,0)}")
    print(f"   VEB 비율:    {veb_ratio:.1f}%")
    print(f"   🏥 종합 위험도: {session_risk}")
    
    all_results.append({
        "reading": reading,
        "beats": beats,
        "peaks": peaks,
        "preds": preds,
        "probas": probas,
        "signal": signal_f,
        "fs": fs,
        "veb_ratio": veb_ratio,
        "session_risk": session_risk,
    })

## 6. 박자별 분류 결과 시각화

In [ ]:
for res in all_results:
    preds   = res["preds"]
    probas  = res["probas"]
    beats   = res["beats"]
    signal  = res["signal"]
    peaks   = res["peaks"]
    fs      = res["fs"]
    reading = res["reading"]
    
    n_show = min(6, len(beats))  # 최대 6개 박자 표시
    fig = plt.figure(figsize=(15, 7))
    gs  = gridspec.GridSpec(2, n_show, height_ratios=[1.5, 1])
    
    # 상단: 전체 파형 + 분류 색상 마커
    ax_full = fig.add_subplot(gs[0, :])
    t = np.arange(len(signal)) / fs
    ax_full.plot(t, signal, color="#2c3e50", lw=0.7, alpha=0.8)
    
    for peak_idx, pred in zip(peaks[:len(preds)], preds):
        ax_full.axvline(
            x=peak_idx / fs,
            color=RISK_COLOR[pred], alpha=0.5, lw=1.5,
        )
        ax_full.scatter(
            peak_idx / fs, signal[peak_idx],
            color=RISK_COLOR[pred], s=50, zorder=5,
        )
    
    ax_full.set_title(
        f"{reading['startTime']} — Beat-level Classification  |  {res['session_risk']}",
        fontsize=11
    )
    ax_full.set_xlabel("Time (s)")
    ax_full.set_ylabel("mV")
    ax_full.grid(alpha=0.3)
    
    # 범례
    for cls, name in LABEL_NAMES.items():
        ax_full.scatter([], [], color=RISK_COLOR[cls],
                        s=50, label=f"{RISK_EMOJI[cls]} {name}")
    ax_full.legend(loc="upper right", fontsize=9)
    
    # 하단: 개별 박자
    t_beat = np.linspace(0, 1000, 360)  # ms
    for col in range(n_show):
        ax = fig.add_subplot(gs[1, col])
        pred   = preds[col]
        conf   = probas[col][pred] * 100
        ax.plot(t_beat, beats[col], color=RISK_COLOR[pred], lw=1.5)
        ax.axhline(0, color="gray", lw=0.5, ls="--")
        ax.set_title(
            f"{RISK_EMOJI[pred]} Beat {col+1}\n"
            f"{LABEL_NAMES[pred]}\n"
            f"Conf: {conf:.0f}%",
            fontsize=8
        )
        ax.set_xlabel("ms", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    fname = f"fitbit_beat_classification_{reading['startTime'][:10]}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fname}")

## 7. 최종 리포트 출력

In [ ]:
print("\n" + "═"*60)
print("  🏥 ECG ARRHYTHMIA REPORT — Fitbit Charge 5")
print("═"*60)
print(f"  분석 일시: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  분석 기록: {len(all_results)}회 측정\n")

for i, res in enumerate(all_results):
    preds = res["preds"]
    unique, counts = np.unique(preds, return_counts=True)
    summary = {int(u): int(c) for u, c in zip(unique, counts)}
    
    print(f"  [{i+1}] {res['reading']['startTime']}")
    print(f"      Fitbit 판정 : {res['reading']['resultClassification']}")
    print(f"      모델 분석   : N={summary.get(0,0)}박  "
          f"S={summary.get(1,0)}박  V={summary.get(2,0)}박")
    print(f"      VEB 비율    : {res['veb_ratio']:.1f}%")
    print(f"      위험도      : {res['session_risk']}")
    print()

print("─"*60)
print("  ⚠️  본 결과는 연구/데모 목적입니다.")
print("      임상적 진단은 반드시 의료전문가와 상담하세요.")
print("═"*60)

---
## 📌 다음 단계 (Roadmap)

- [ ] **실제 Fitbit 토큰** 발급 후 본인 ECG 데이터로 실행
- [ ] **MIT-BIH 실제 데이터**로 모델 재학습 → 정확도 향상
- [ ] **모바일 앱** 개발 (FastAPI 백엔드 + React Native 프론트)
- [ ] **5분 간격 자동 분석** 스케줄러 추가
- [ ] **Samsung Galaxy Watch / Apple Watch** 어댑터 추가